<a href="https://colab.research.google.com/github/blufury/Class-projects/blob/master/sam_bennett_hw_tokenization_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Part 0 - Setup

In [3]:
#Imports
import re, random
from collections import Counter
import torch
import torch.nn as nn
import tiktoken

random.seed(42)
torch.manual_seed(42)

path = "/content/the-verdict.txt"
with open(path, "r") as f:
  text = f.read()

print("Character Count:", len(text))
print("Line Count:", text.count("\n")+1)
print("First 300 characters:")
print(text[:300])

Character Count: 20479
Line Count: 165
First 300 characters:
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would ha


## Part 1 - Regex Tokenization (Simple Tokenizer)

In [4]:
token_re = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def regex_tokenize(text):
  return token_re.findall(text)

tokens = regex_tokenize(text)
print("Total Token Count:", len(tokens))
freq = Counter(tokens)
print("Unique Tokens:", len(freq))

print("Top 30 Tokens:")
for tok, c in freq.most_common(30):
  print(f"{tok!r}\t{c}")

def not_word(tok):
  return re.fullmatch(r"[^\w]+", tok) is not None

def is_word(tok):
  return any(ch.isalpha() for ch in tok)

punct = sum(1 for t in tokens if not_word(t))
word = sum(1 for t in tokens if is_word(t))

print("Punctuation-like Fraction:", punct/len(tokens))
print("Word-like Fraction:", word/len(tokens))

Total Token Count: 4827
Unique Tokens: 1148
Top 30 Tokens:
'-'	232
','	229
'.'	212
'the'	167
'"'	137
'I'	121
'of'	98
'to'	97
"'"	94
'a'	82
'and'	75
'was'	69
'it'	68
'he'	64
'his'	61
'that'	55
'had'	53
'in'	40
'my'	40
's'	38
'me'	37
'him'	33
'with'	32
't'	31
'on'	27
'have'	25
'you'	25
'as'	24
'?'	24
'!'	24
Punctuation-like Fraction: 0.2073751812720116
Word-like Fraction: 0.7926248187279884


## Part 2 - Vocabulary + Encode/Decode + Special Tokens

In [5]:
special_tokens = ["<|unk|>", "<|endoftext|>", "[PAD]"]

vocab = {st:i for i, st in enumerate(special_tokens)}
for tok in freq:
  if tok not in vocab:
    vocab[tok] = len(vocab)
reverse_vocab = {i:t for t,i in vocab.items()}

UNK = vocab["<|unk|>"]
EOT = vocab["<|endoftext|>"]
PAD = ["[PAD]"]

def encode(tok_list, vocab=vocab):
  return [vocab.get(t, UNK) for t in tok_list]

def decode(ids, reverse_vocab=reverse_vocab):
  out = []
  prev = ""
  for i in ids:
    tok = reverse_vocab.get(i, "<|unk|>")
    if not out:
      out.append(tok)
    else:
      if re.fullmatch(r"[^\w\s]", tok):
        out.append(tok)
      else:
        if prev in ["(", "[", "{", '"', "'"]:
          out.append(tok)
        else:
          out.append(" " + tok)
    prev = tok
  return "".join(out)

#1 round-trip
rt = text[:250]
print("Round-Trip Snippet:\n", decode(encode(regex_tokenize(rt))))

#2 oov test
oov_sentence = "WUBBAWUBBA WAKKAFLOCKA RINGADING ROGERDODGER RATATATATA"
oov_ids = encode(regex_tokenize(oov_sentence))
print("OOV IDs:", oov_ids)
print("OOV Decoded:", decode(oov_ids))

#3 end token test
end_ids = encode(regex_tokenize("End token Test.")) + [EOT]
print("End-token Decoded:", decode(end_ids))

Round-Trip Snippet:
 I HAD always thought Jack Gisburn rather a cheap genius-- though a good fellow enough-- so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on
OOV IDs: [0, 0, 0, 0, 0]
OOV Decoded: <|unk|> <|unk|> <|unk|> <|unk|> <|unk|>
End-token Decoded: <|unk|> <|unk|> <|unk|>. <|endoftext|>


## Part 3 - BPE Tokenization with Tiktoken (GPT-2)

In [8]:
tiktok = tiktoken.get_encoding("gpt2")
bpe_ids = tiktok.encode(text)

print("Total Token Count:", len(bpe_ids))
print("Unique Token IDs", len(set(bpe_ids)))

def excerpt(start, length=320):
  return text[start:start+length]

beginning = excerpt(0, 320)
middle = excerpt(max(0, len(text)//2-160), 320)
end = excerpt(max(0, len(text) - 320), 320)

for name, ex in [("Beginning", beginning), ("Middle", middle), ("End", end)]:
  regex_count = len(regex_tokenize(ex))
  bpe_count = len(tiktok.encode(ex))
  print(f"\n{name} excerpt counts:")
  print("  chars:", len(ex))
  print("  regex tokens:", regex_count)
  print('  BPE tokens:', bpe_count)
  print("  decoded round-trip preview:")
  print(tiktok.decode(tiktok.encode(ex))[:200], "...")

Total Token Count: 5145
Unique Token IDs 1416

Beginning excerpt counts:
  chars: 320
  regex tokens: 71
  BPE tokens: 74
  decoded round-trip preview:
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a ...

Middle excerpt counts:
  chars: 320
  regex tokens: 78
  BPE tokens: 77
  decoded round-trip preview:
estic economy. And whenever my wonder paid the expected tribute he said, throwing out his chest a little: "Yes, I really don't see how people manage to live without that."

Well--it was just the end o ...

End excerpt counts:
  chars: 320
  regex tokens: 79
  BPE tokens: 81
  decoded round-trip preview:
n the one thing that brings me anywhere near him is that I knew enough to leave off?"

He stood up and laid his hand on my shoulder with a laugh. "Only the irony of it is that I _am_ still painting--s ...


Part 4 - Sliding-window Dataset for Next-token Prediction (BPE IDs)

In [9]:
def make_window(token_ids, T=32, S=16):
  xs, ys = [], []
  for i in range(0, len(token_ids) - (T+1) + 1, S):
    xs.append(token_ids[i:i+T])
    ys.append(token_ids[i+1:i+T+1])
  return torch.tensor(xs, dtype=torch.long), torch.tensor(ys, dtype=torch.long)

T, S = 32, 16
xs, ys = make_window(bpe_ids, T=T, S=S)
print("xs shape:", xs.shape)
print("ys shape:", ys.shape)

i = 0
print("Decoded x context:", tiktok.decode(xs[i].tolist()))
print("Decoded y targets:", tiktok.decode(ys[i].tolist()))

xs shape: torch.Size([320, 32])
ys shape: torch.Size([320, 32])
Decoded x context: I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that,
Decoded y targets:  HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in


## Part 5 - Embeddings: Token + Positional

In [10]:
V = getattr(tiktok, "n_vocab", 256)
D = 64
T = 32

token_emb = nn.Embedding(V, D)
pos_emb = nn.Embedding(T, D)

B = 2
x_batch = torch.zeros((B, T), dtype=torch.long)

token = int(bpe_ids[100] if len(bpe_ids) > 150 else bpe_ids[0])
x_batch[0, 0] = token
x_batch[0, 5] = token
x_batch[1, 0] = token
x_batch[1, 10] = token

tok = token_emb(x_batch)         #[B,T,D]
pos = pos_emb(torch.arange(T))   #[T,D]
inp = tok + pos                  #[B,T,D]

print("x_batch shape:", x_batch.shape)
print("tok shape:", tok.shape)
print("pos shape:", pos.shape)
print("inp shape:", inp.shape)

print("Same token id:", token)
print("tok[0,0] == tok[0,5]:", torch.allclose(tok[0,0], tok[0,5]))
print("tok[1,0] == tok[1,10]:", torch.allclose(tok[1,0], tok[1,10]))

print("Final embedding difference by positoin:")
print("inp[0,0] == inp[0,5]:", torch.allclose(inp[0,0], inp[0,5]))
print("inp[1,0] == inp[1,10]:", torch.allclose(inp[1,0], inp[1,10]))

x_batch shape: torch.Size([2, 32])
tok shape: torch.Size([2, 32, 64])
pos shape: torch.Size([32, 64])
inp shape: torch.Size([2, 32, 64])
Same token id: 5469
tok[0,0] == tok[0,5]: True
tok[1,0] == tok[1,10]: True
Final embedding difference by positoin:
inp[0,0] == inp[0,5]: False
inp[1,0] == inp[1,10]: False


## Part 6 - Reflection

In [ ]:
#See attached word file